In [1]:
import xdem, geoutils as gu
import numpy as np

In [3]:
# co-register and dDEM for 2025 - 2010
ref = xdem.DEM("../data/rasters/cfc_elevfilled_2010_5m.tif")
if ref.nodata is None: ref.set_nodata(-9999)

tba = xdem.DEM("../data/rasters/Canwell_4Aug25_filled1m.tif")
if tba.nodata is None: tba.set_nodata(-9999)
tba = tba.reproject(ref, resampling="average")

# drop thinning basin to fit on stable-ish ground
basin_mask = gu.Vector("../data/outlines/ncanwell_longbasin_nolap.gpkg").create_mask(ref)
inlier = ~basin_mask

# purer (fewer pixels) — also drop actively deforming north slope:
# slope_mask = gu.Vector("../outlines/ncanwell_longslope_nolap.gpkg").create_mask(ref)
# inlier = ~(basin_mask | slope_mask)

nk = xdem.coreg.NuthKaab()
nk.fit(ref, tba, inlier_mask=inlier)        # if arg-name error: reference_elev=ref, to_be_aligned_elev=tba
tba_c = nk.apply(tba)

def stats(a, b, label):
    d = (a - b).data
    nmad = xdem.spatialstats.nmad(d)        # fallback: 1.4826*np.ma.median(np.ma.abs(d-np.ma.median(d)))
    print(f"{label}: median {np.ma.median(d):+.2f} m | NMAD {nmad:.2f} m")

stats(tba,   ref, "before")
stats(tba_c, ref, "after ")
print("coreg meta:", nk.meta)               # solved dx, dy, dz live in here

tba_c.save("../data/rasters/canwell_2025_coreg_on2010grid.tif")
(tba_c - ref).save("../data/rasters/canwell_ddem_2025_2010_coreg_5m.tif")

/home/samuelnwalters/miniconda3/envs/gd_env/lib/python3.10/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(
/tmp/ipykernel_984/1603728968.py:22: DeprecationWarning: Call to deprecated function 'nmad'. Xdem.spatialstats.nmad is being deprecated in favor of geoutils.stats.nmad. This functionality will be removed in version 0.4.
  nmad = xdem.spatialstats.nmad(d)        # fallback: 1.4826*np.ma.median(np.ma.abs(d-np.ma.median(d)))


before: median +14.09 m | NMAD 3.03 m


/tmp/ipykernel_984/1603728968.py:22: DeprecationWarning: Call to deprecated function 'nmad'. Xdem.spatialstats.nmad is being deprecated in favor of geoutils.stats.nmad. This functionality will be removed in version 0.4.
  nmad = xdem.spatialstats.nmad(d)        # fallback: 1.4826*np.ma.median(np.ma.abs(d-np.ma.median(d)))


after : median -0.51 m | NMAD 2.94 m
coreg meta: {'inputs': {'random': {'subsample': 500000.0, 'random_state': None}, 'fitorbin': {'fit_or_bin': 'bin_and_fit', 'fit_func': <function _nuth_kaab_fit_func at 0x7c17f1198310>, 'fit_optimizer': <function curve_fit at 0x7c17fdfecc10>, 'bin_sizes': 72, 'bin_statistic': <function nanmedian at 0x7c19175f0b30>, 'nd': 1, 'bias_var_names': ['aspect']}, 'iterative': {'max_iterations': 10, 'tolerance': 0.001}, 'specific': {}, 'affine': {'apply_vshift': True}}, 'outputs': {'affine': {'shift_x': np.float64(-1.2880443702157183), 'shift_y': np.float64(0.030602962898653902), 'shift_z': -14.710176327373347}, 'random': {'subsample_final': 500000}}}


/tmp/ipykernel_984/1603728968.py:29: DeprecationWarning: Call to deprecated function 'save'. The function .save() will be soon deprecated, use .to_file() instead. This functionality will be removed in version 0.3.0.
  tba_c.save("../data/rasters/canwell_2025_coreg_on2010grid.tif")
/tmp/ipykernel_984/1603728968.py:30: DeprecationWarning: Call to deprecated function 'save'. The function .save() will be soon deprecated, use .to_file() instead. This functionality will be removed in version 0.3.0.
  (tba_c - ref).save("../data/rasters/canwell_ddem_2025_2010_coreg_5m.tif")


In [5]:
# co-register and clip 2010 - 2000
ref = xdem.DEM("../data/zarchive2010/cfc_elevfilled_2010_5m.tif")

dem00 = xdem.DEM("../data/zarchive2000/CanFelCas_2000filled_5m.tif")   # confirm name
if dem00.nodata is None: dem00.set_nodata(-9999)
dem00 = dem00.reproject(ref, resampling="average")                # ref = 2010, already loaded

basin_mask = gu.Vector("../data/outlines/ncanwell_longbasin_nolap.gpkg").create_mask(ref)
inlier = ~basin_mask

nk2 = xdem.coreg.NuthKaab()
nk2.fit(ref, dem00, inlier_mask=inlier)
dem00_c = nk2.apply(dem00)

stats(dem00,   ref, "before")     # your stats() fn from the target cell
stats(dem00_c, ref, "after ")
print("coreg meta:", nk2.meta["outputs"]["affine"])

(ref - dem00_c).to_file("../data/rasters/cfc_ddem_2010_2000_coreg_5m.tif")   # past-dDEM = 2010 - 2000

/home/samuelnwalters/miniconda3/envs/gd_env/lib/python3.10/site-packages/geoutils/raster/geotransformations.py:101: UserWarning: Output projection, bounds and grid size are identical -> returning self (not a copy!)
  warnings.warn("Output projection, bounds and grid size are identical -> returning self (not a copy!)")
/tmp/ipykernel_984/1603728968.py:22: DeprecationWarning: Call to deprecated function 'nmad'. Xdem.spatialstats.nmad is being deprecated in favor of geoutils.stats.nmad. This functionality will be removed in version 0.4.
  nmad = xdem.spatialstats.nmad(d)        # fallback: 1.4826*np.ma.median(np.ma.abs(d-np.ma.median(d)))


before: median +0.54 m | NMAD 2.73 m


/tmp/ipykernel_984/1603728968.py:22: DeprecationWarning: Call to deprecated function 'nmad'. Xdem.spatialstats.nmad is being deprecated in favor of geoutils.stats.nmad. This functionality will be removed in version 0.4.
  nmad = xdem.spatialstats.nmad(d)        # fallback: 1.4826*np.ma.median(np.ma.abs(d-np.ma.median(d)))


after : median +0.10 m | NMAD 2.59 m
coreg meta: {'shift_x': np.float64(-1.7253904197526606), 'shift_y': np.float64(0.00422738922667651), 'shift_z': -0.5101216565337836}
